In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pywt

## Config

In [10]:
IN_PATH  = "../data/NF-UNSW-NB15-v3.csv"
label_col = 'Attack'
wavelet_name = 'cmor3.5-1.0'
max_samples = 2000
scales = np.arange(1, 128)

In [11]:
#Load the data
df = pd.read_csv(IN_PATH)

In [12]:
attack_types = df["Attack"].unique()

In [13]:
rows = []

In [14]:
for attack_type in attack_types:
    df_attack = df[df["Attack"] == attack_type]
    
    n_samples = min(len(df_attack), max_samples)
    df_attack = df_attack.iloc[:n_samples]
    df_attack = df_attack.sort_values(by="FLOW_START_MILLISECONDS")
    
    iat_signal = df_attack["SRC_TO_DST_IAT_AVG"].replace([np.inf, -np.inf], np.nan).dropna().values
    if len(iat_signal) == 0:
        continue
    
    iat_signal = (iat_signal - np.mean(iat_signal)) / np.std(iat_signal)
    
    coefficients, frequencies = pywt.cwt(iat_signal, scales, wavelet_name)
    
    coeff_magnitude = np.abs(coefficients)
    
    reduced_features = np.mean(coeff_magnitude, axis=1)
    
    reduced_features = (reduced_features - np.mean(reduced_features)) / np.std(reduced_features)
    
    row_dict = {"attack_type": attack_type}
    for i, val in enumerate(reduced_features):
        row_dict[f"cwt_mean_scale_{i}"] = val
    
    rows.append(row_dict)

In [15]:
df_cwt = pd.DataFrame(rows)

In [16]:
df_cwt.to_csv("cwt_iat_avg_all_attacks.csv", index=False)

In [18]:
df = pd.read_csv("cwt_iat_avg_all_attacks.csv")

In [19]:
df.head()

,attack_type,cwt_mean_scale_0,cwt_mean_scale_1,cwt_mean_scale_2,cwt_mean_scale_3,cwt_mean_scale_4,cwt_mean_scale_5,cwt_mean_scale_6,cwt_mean_scale_7,cwt_mean_scale_8,...,cwt_mean_scale_117,cwt_mean_scale_118,cwt_mean_scale_119,cwt_mean_scale_120,cwt_mean_scale_121,cwt_mean_scale_122,cwt_mean_scale_123,cwt_mean_scale_124,cwt_mean_scale_125,cwt_mean_scale_126
0,Benign,-3.013499,-2.801846,-2.533462,-2.330793,-2.139166,-2.008894,-1.919978,-1.825699,-1.760133,...,1.046134,1.139519,1.102965,1.125107,1.200614,1.206513,1.197820,1.282992,1.357523,1.303799
1,Fuzzers,-2.250552,-2.019613,-1.752103,-1.588368,-1.622672,-1.656309,-1.571706,-1.450404,-1.509844,...,1.451566,1.443569,1.475044,1.468418,1.444462,1.466631,1.489906,1.501198,1.495692,1.522205
2,Exploits,-2.838223,-2.229535,-1.900871,-1.725769,-1.519968,-1.451248,-1.539646,-1.480905,-1.300227,...,1.547882,1.845551,1.310303,1.162813,1.476364,1.855171,1.589458,1.081139,1.124624,1.658078
3,Backdoor,-3.574099,-2.736698,-2.222656,-2.149075,-1.869740,-1.792819,-1.716424,-1.630447,-1.491292,...,1.393625,1.340344,1.417724,1.715414,1.929867,1.496358,1.685754,2.222531,2.023633,1.981039
4,Reconnaissance,-2.437127,-2.159356,-2.055877,-2.047714,-1.997688,-2.008002,-1.858380,-1.772647,-1.744458,...,1.176563,1.271993,1.369012,1.409317,1.451749,1.584683,1.636097,1.698535,1.765306,1.923704
